# Tokenizer Basics

> We have seen that all computations inside a neural network are numerical operations -- matrix multiplication, activation functions, gradient updates -- and that the inputs and outputs they accept and produce are all numbers. Yet the language we use every day is made of words, not numbers.
>
> In this section, we build a Tokenizer from scratch. We will implement three approaches in turn -- character-level, word-level, and subword-level -- observing the trade-offs each one introduces, and understanding why subword-level has become the shared choice of modern large language models.

The mechanism that converts text into numbers is called a Tokenizer. A Tokenizer first splits text into small pieces, then assigns an integer ID to each piece. These small pieces are called tokens, and the integer IDs are called token IDs.

For example, the sentence `"the cat sat"` is split into three tokens: `["the", "cat", "sat"]`. A lookup table (called the vocabulary) then maps them to IDs `[5, 1, 3]`. After this processing, the text has become a sequence of integers that the model can work with.

There are several possible granularities for splitting. Character-level splitting treats each letter as a token -- for instance, `"cat"` becomes `["c", "a", "t"]`, which produces a very long sequence. Word-level splitting treats each word as a token -- `"the cat sat"` becomes `["the", "cat", "sat"]`, which shortens the sequence but inflates the vocabulary, and any unseen word cannot be handled. Subword-level splitting strikes a balance between these two, and is the approach most widely adopted by modern large language models.

In [1]:
# First, use a real tokenizer to get a feel for how text becomes tokens
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "the cat sat on the mat"
ids = tokenizer.encode(text)
tokens = [tokenizer.decode([i]) for i in ids]

print(f"Original text: {text}")
print(f"token: {tokens}")
print(f"token ID: {ids}")

Original text: the cat sat on the mat
token: ['the', ' cat', ' sat', ' on', ' the', ' mat']
token ID: [1169, 3797, 3332, 319, 262, 2603]


## Key Questions for This Section

After studying this section, you should be able to answer:

1. What does a Tokenizer do?
2. What are the advantages and disadvantages of character-level splitting?
3. What are the advantages and disadvantages of word-level splitting?
4. Why is subword-level the default choice for modern LLMs?

## 1. Tokens and Tokenizers

As mentioned earlier, a token is the smallest unit of text that a model processes. Depending on the splitting approach, a token can be a character, a word, or a subword piece:

```text
Character-level:  "cat" → ["c", "a", "t"]
Word-level:       "the cat" → ["the", "cat"]
Subword-level:    "playing" → ["play", "ing"]
```

A Tokenizer exposes two core operations: `encode` (text → token ID sequence) and `decode` (token ID sequence → text).

```text
Text       --encode-->  token ID sequence
ID sequence --decode-->  text
```

Note that token IDs are just labels; they do not represent magnitude. A token with ID 12 is not "greater" than one with ID 3 -- it simply occupies the 12th position.

Let us prepare a small corpus and implement each of the three splitting approaches in turn.

**Experiment Corpus**

A Tokenizer needs to gather statistics from a corpus and build a vocabulary, so we first prepare a small collection of text. We use English here because English has spaces as natural word boundaries, making it easy to observe the splitting process. Once the principles are clear, the Chinese case simply involves more subtle boundary markers that require finer rules.

In [2]:
# Our corpus -- simple sentences with deliberate repetition
# This way, BPE merges will clearly show how high-frequency pairs get combined
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print(f"Corpus contains {len(corpus)} sentences")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

total_chars = sum(len(s) for s in corpus)
print(f"\nTotal characters: {total_chars}")
print(f"Total words (split by space): {sum(len(s.split()) for s in corpus)}")

Corpus contains 10 sentences
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends

Total characters: 175
Total words (split by space): 46


**How Computers Represent Text**

The `the cat` displayed on screen is, inside the computer, a sequence of numeric codes (using ASCII as an example):

```
Text:    t    h    e         c    a    t
         ↓    ↓    ↓    ↓    ↓    ↓    ↓
ASCII:  116  104  101  32   99   97   116
```

Converting text to numbers is something computers already do. The question a Tokenizer answers is not "can we convert it?" but rather "how fine should we split it?":

- Character-level: each letter is one token, the finest granularity.
- Word-level: each word is one token, the coarsest granularity.
- Subword-level: common pieces are one token, somewhere in between.

We will try each approach in turn, focusing on the same question each time: what does this splitting method solve, and what new problem does it introduce?

## 2. Character-Level Tokenizer

The character-level approach is the most straightforward: each character is a token.

```
"the cat"
  ↓ split by character
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↓ look up IDs
[5, 12, 3, 0, 6, 1, 8]
```

The vocabulary is simply a token-to-ID mapping table. In the character-level setting, each token is a single character. The advantages are clear: the implementation is simple, the vocabulary is stable (only a few dozen characters are needed), and there is essentially no such thing as an "unseen character" -- any Unicode character can be included.

Let us implement a character-level Tokenizer in Python.

In [9]:
class CharTokenizer:
    """
    Character-level Tokenizer: one character = one token
    Two core methods: encode and decode
    """

    def __init__(self):
        # stoi = string-to-id, maps characters to numbers
        # itos = id-to-string, maps numbers back to characters
        self.stoi = {}
        self.itos = {}

    def train(self, texts):
        """
        Training: collect all characters that appear in the corpus, build vocabulary

        Steps:
        1. Join all texts into one long string
        2. Use set to deduplicate → get all unique characters
        3. Sort → assign each character a unique ID
        """
        all_text = "".join(texts)
        chars = sorted(list(set(all_text)))

        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}

        print(f"=== Character-level Tokenizer training complete ===")
        print(f"Vocabulary size: {len(self.stoi)} tokens")
        print(f"Vocabulary items: {chars}")

    def encode(self, text):
        """Encode: text → token ID list"""
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        """Decode: token ID list → text"""
        return "".join([self.itos[i] for i in ids])

In [4]:
# Train and test
char_tokenizer = CharTokenizer()
char_tokenizer.train(corpus)

# Test with one sentence
text = "the cat"
ids = char_tokenizer.encode(text)
recovered = char_tokenizer.decode(ids)

print(f"\n=== Encode/Decode Test ===")
print(f"Original text:     '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation: {len(text)} characters in the original → {len(ids)} tokens produced, same count.")
print(f"Each character becomes one token, so there is no compression.")

=== Character-level Tokenizer training complete ===
Vocabulary size: 20 tokens
Vocabulary items: [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y']

=== Encode/Decode Test ===
Original text:     'the cat'
Token IDs: [16, 7, 4, 0, 2, 1, 16]
Decoded back: 'the cat'

Key observation: 7 characters in the original → 7 tokens produced, same count.
Each character becomes one token, so there is no compression.


### Limitations of Character-Level Splitting

The character-level approach has two notable drawbacks.

First, the sequences are too long. `"the cat"` is 7 characters, producing 7 tokens. A 1000-character article would require 1000 tokens. The Self-Attention mechanism is highly sensitive to sequence length -- both computation and memory consumption grow quickly as the sequence gets longer.

Second, semantics are fragmented. In natural language, "cat" is a complete concept, but character-level splitting breaks it into three separate letters: c, a, t. The model must learn from large amounts of data that these three letters frequently co-occur and together represent the concept of "cat", which adds to the learning burden.

Both of these problems -- sequences being too long and semantics being fragmented -- point toward the same improvement: can we split text using larger units? A natural idea is to split by words.

## 3. Word-Level Tokenizer

A word-level Tokenizer splits text by spaces, where each word is a token.

```
"the cat sat on the mat"
  ↓ split by space
['the', 'cat', 'sat', 'on', 'the', 'mat']
  ↓ look up IDs
[0, 1, 2, 3, 0, 4]
```

Compared to character-level, the sequence length is significantly reduced. `"the cat"` needs only 2 tokens, whereas character-level needs 7.

However, the word-level approach has a fundamental problem: what happens when it encounters a word outside the vocabulary? Suppose the vocabulary includes "cat" but not "cats" -- the Tokenizer cannot assign a valid ID to "cats". This situation is called OOV (out of vocabulary). Real-world text is full of variations -- plural forms, tense changes, derived words, typos, and internet slang. If every variant is included in the vocabulary separately, the vocabulary swells to an unmanageable size; if not, OOV occurs frequently. This is a difficult trade-off.

In [5]:
class WordTokenizer:
    """
    Word-level Tokenizer: split by space, one word = one token
    Same structure as CharTokenizer, but splitting granularity
    changes from "character" to "word"
    """

    def __init__(self):
        self.stoi = {}
        self.itos = {}

    def train(self, texts):
        """Collect all words that appear in the corpus"""
        all_words = set()
        for text in texts:
            words = text.split()
            all_words.update(words)

        all_words = sorted(list(all_words))
        self.stoi = {w: i for i, w in enumerate(all_words)}
        self.itos = {i: w for i, w in enumerate(all_words)}

        print(f"=== Word-level Tokenizer training complete ===")
        print(f"Vocabulary size: {len(self.stoi)} words")
        print(f"Vocabulary items: {all_words}")

    def encode(self, text):
        """Split text by space, look up each word"""
        return [self.stoi[w] for w in text.split()]

    def decode(self, ids):
        """Look up IDs to get words, join with spaces"""
        return " ".join([self.itos[i] for i in ids])

In [6]:
# Train and test
word_tokenizer = WordTokenizer()
word_tokenizer.train(corpus)

text = "the cat sat on the mat"
ids = word_tokenizer.encode(text)
recovered = word_tokenizer.decode(ids)

print(f"\n=== Encode/Decode Test ===")
print(f"Original text:     '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation: {len(text.split())} words in the original → {len(ids)} tokens produced.")
print(f"The same sentence needs 7 tokens at character-level but only 6 at word-level.")

=== Word-level Tokenizer training complete ===
Vocabulary size: 20 words
Vocabulary items: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

=== Encode/Decode Test ===
Original text:     'the cat sat on the mat'
Token IDs: [19, 2, 17, 16, 19, 14]
Decoded back: 'the cat sat on the mat'

Key observation: 6 words in the original → 6 tokens produced.
The same sentence needs 7 tokens at character-level but only 6 at word-level.


In [8]:
# Word-level tokenizer behavior when encountering unseen words (OOV)
print("=== OOV (Out Of Vocabulary) Demo ===")
print(f"Current vocabulary: {list(word_tokenizer.stoi.keys())}")
print()

try:
    test_text = "the elephant sat"
    print(f"Trying to encode: '{test_text}'")
    ids = word_tokenizer.encode(test_text)
    print(f"Result: {ids}")
except KeyError as e:
    words = test_text.split()
    for w in words:
        if w in word_tokenizer.stoi:
            print(f"  '{w}' → ID {word_tokenizer.stoi[w]}  (in vocabulary)")
        else:
            print(f"  '{w}' → not in vocabulary, raises KeyError")
    print(f"\nConclusion: the unseen word 'elephant' causes encoding to fail.")
    print(f"In practice, it is impossible to pre-load every word into the vocabulary.")

=== OOV (Out Of Vocabulary) Demo ===
Current vocabulary: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

Trying to encode: 'the elephant sat'
  'the' → ID 19  (in vocabulary)
  'elephant' → not in vocabulary, raises KeyError
  'sat' → ID 17  (in vocabulary)

Conclusion: the unseen word 'elephant' causes encoding to fail.
In practice, it is impossible to pre-load every word into the vocabulary.


## 4. Subword-Level Tokenizer

Both character-level and word-level approaches have their problems. The subword-level approach tries to strike a balance between them. A comparison of the three:

| Approach | Vocab Size | Seq Length | OOV Issue | Characteristics |
|:---|:---|:---|:---|:---|
| Character-level | Very small | Very long | Almost none | Too fine, semantics fragmented |
| Word-level | Very large | Very short | Frequent | Too coarse, inflexible |
| Subword-level | Controllable | Moderate | Rare | A compromise between the two |

The effect of subword-level splitting becomes clear with a few examples:

```
"the cat sat"      → [the, cat, sat]        ← common words kept whole
"cats and dogs"    → [cat, s, and, dog, s]  ← uncommon plural forms split
"unbelievable"     → [un, believ, able]      ← rare word split into known pieces
```

The idea is straightforward: high-frequency words are kept whole in the vocabulary, while low-frequency or unseen words are split into smaller pieces that already exist in the vocabulary. This way, the vocabulary does not inflate like word-level, and new words can usually be decomposed into known subword pieces.

### Introduction to BPE

BPE (Byte Pair Encoding) is one of the most widely used subword algorithms today. Its core operation is a single step: find the most frequently occurring pair of adjacent tokens and merge them into a new token. Then repeat.

Looking at a concrete example:

```
Initial: one token per character
['t', 'h', 'e', ' ', 'c', 'a', 't', ...]
        ↓ count occurrences of all adjacent pairs
('t', 'h') has the highest frequency
        ↓ merge them
New token: 'th'
        ↓ keep counting, keep merging...
```

After multiple rounds of iteration, the vocabulary grows from the initial character set to include common pieces like th, the, ing, tion, and so on. In the next section, we will implement BPE from scratch, showing every step of the counting and merging process in full detail.

## Summary

What we learned in this section:

- A Tokenizer provides bidirectional conversion between text and token ID sequences -- via encode and decode
- The vocabulary is a token-to-ID mapping table; each token has a unique ID, and the ID itself does not represent magnitude
- Character-level Tokenizer: small, stable vocabulary with almost no OOV issues, but long sequences and fragmented semantics
- Word-level Tokenizer: short sequences, but vocabulary prone to inflation and unable to handle out-of-vocabulary words (OOV)
- Subword-level Tokenizer: strikes a balance between vocabulary size and sequence length, keeping high-frequency words whole while splitting low-frequency words into known pieces; it is the default choice for modern large language models

In the next section, we will implement the BPE algorithm from scratch and see how a subword vocabulary is built step by step starting from characters.

## Exercises

Three exercises covering two categories of content:

1. **Core mechanism**: The essence of a Tokenizer is the text ↔ token ID mapping; doing it by hand helps deepen understanding.
2. **Practical usage**: Special tokens, attention masks, and batching variable-length sequences -- these are all standard operations in model training and inference.

> **About AI assistance**: You can ask an AI for hints, step-by-step reasoning, or a direction check, but it is not recommended to have the AI complete the exercises for you. The difference between doing it yourself and simply reading the answer is significant.

### Exercise 1: How encode Works

Looking up tokens in a table and converting them to IDs -- this is the core operation of a Tokenizer.

**Hint**: `stoi[token]` is the lookup -- it maps a token to its corresponding ID.

In [1]:
# Exercise 1: core mechanism -- fill in the encode step
stoi = {"the": 0, "cat": 1, "sat": 2}
tokens = ["the", "cat", "sat"]

# TODO: replace the triple-quoted content below with your code
ids = [stoi[token] for token in tokens]

assert not isinstance(ids, str), "Please replace the triple-quoted placeholder first"
assert ids == [0, 1, 2], ids
print("Exercise 1 passed: you understand that encode is token → ID lookup")

Exercise 1 passed: you understand that encode is token → ID lookup


### Exercise 2: Using Special Tokens

Model input is not limited to the text itself. `<BOS>` marks the beginning of a sequence, and `<EOS>` marks the end -- the model relies on these symbols to identify sentence boundaries.

**Hint**: A common input format is `[BOS] + text tokens + [EOS]`, with boundary symbols on both ends.

In [3]:
# Exercise 2: special tokens fill-in
stoi = {"<BOS>": 0, "<EOS>": 1, "the": 2, "cat": 3}
tokens = ["the", "cat"]

# TODO: replace the triple-quoted content below with your code
input_ids = [stoi[token] for token in ["<BOS>"] + tokens + ["<EOS>"]]

assert not isinstance(input_ids, str), "Please replace the triple-quoted placeholder first"
assert input_ids == [0, 2, 3, 1], input_ids
print("Exercise 2 passed: you understand the role of special tokens in input sequences")

Exercise 2 passed: you understand the role of special tokens in input sequences


### Exercise 3: Padding and Attention Masks

Sentences within a batch usually vary in length. To form a uniform tensor, shorter sentences are padded with `<PAD>` at the end, while an attention mask marks which positions are real tokens and which are padding.

**Hint**: Mask value is 1 for real token positions and 0 for padding positions. The model uses this to ignore padding when computing Attention.

In [5]:
# Exercise 3: padding + attention mask fill-in
PAD_ID = 0
ids = [5, 6, 7]
max_len = 5

# TODO: replace the triple-quoted content below with your code
padded_ids = (ids + [PAD_ID] * max_len)[:max_len]
attention_mask = [1 if padded_id != 0 else 0 for padded_id in padded_ids]

assert not isinstance(padded_ids, str), "Please replace the padded_ids placeholder first"
assert not isinstance(attention_mask, str), "Please replace the attention_mask placeholder first"
assert padded_ids == [5, 6, 7, 0, 0], padded_ids
assert attention_mask == [1, 1, 1, 0, 0], attention_mask
print("Exercise 3 passed: you understand how padding and attention masks work together")

Exercise 3 passed: you understand how padding and attention masks work together
